# Exploration

Using the font fix, then printing the amount of sheets

In [12]:
from openpyxl.styles.fonts import Font
Font.family.max = 100          # MUST run before load_workbook

from openpyxl import load_workbook

wb = load_workbook("../data/raw/Nutrition141025NEW.xlsx",
                   read_only=True, data_only=True)
print(len(wb.sheetnames))      # expect 94

94


Discovery of the sheet names

In [13]:
for i, name in enumerate(wb.sheetnames):
    print(i, repr(name))

0 'Weight J'
1 'Weight J (2)'
2 'Weight OR'
3 'Costs'
4 'Blank Simplified'
5 'Blank Week'
6 '14Oct24'
7 '4Nov24'
8 'Sheet1 (2)'
9 '19Nov24'
10 '25Nov24'
11 '2Dec24'
12 '9Dec24'
13 '16Dec24'
14 '30Dec24'
15 '6Jan24'
16 '20Jan24'
17 '27Jan25'
18 '03Feb25'
19 '10Feb25'
20 '17Feb25'
21 '24Feb25'
22 '03Mar25'
23 '10Mar25'
24 '17Mar25'
25 '24Mar25'
26 '31Mar25'
27 '07Apr25'
28 '14Apr25'
29 '21Apr25'
30 '28Apr25'
31 '05May25'
32 '13May25'
33 '19May25'
34 '26May25'
35 '02June25'
36 '09June25'
37 '30JUN25'
38 '07JUL25'
39 '14JUL25'
40 'Blank Schedule (2)'
41 'Blank (2)'
42 '11AUG25'
43 '18AUG25'
44 '25AUG25'
45 '01SEP25'
46 '08SEP25'
47 '15SEP25'
48 '06OCT25'
49 '13OCT25'
50 '20OCT25'
51 '27OCT25'
52 '03NOV25'
53 'IGNORE'
54 '10NOV25'
55 '17NOV25'
56 '24NOV25'
57 '01DEC25'
58 '08DEC25'
59 '29DEC25'
60 '05JAN26'
61 '12JAN26'
62 '19JAN26'
63 '26Jan26'
64 '02Feb26'
65 '09Feb26'
66 '16Feb26'
67 '23Feb26'
68 '02Mar26'
69 '09Mar26'
70 '16Mar26'
71 '23Mar26'
72 '30Mar26'
73 '07APR26'
74 '13APR26'
75 '

Creating multi-sheet row finder function

In [14]:
def find_header_rows(sheet_name, target="Time"): #defining our function so we can reuse, sets both parameters of sheet_name and target="Time" as default, though we can use this to find any value.
    wb = load_workbook("../data/raw/Nutrition141025NEW.xlsx",
                   read_only=True, data_only=True) #we have to reload the workbook here because we can only read each sheet once

    ws = wb[sheet_name] #selecting our sheet 

    for row_number, cell_values in enumerate(ws.iter_rows(values_only=True),start=1): #loop which runs row by row, enumerating each cell with row number and cell value
        time_columns = [] #starting an empty list
        for column_number, value in enumerate(cell_values): #second loop which enumerates the column number and value for each given cell
            if str(value).strip() == target: #takes the stripped output of the string and if it is equal to target parameter...
                time_columns.append(column_number) #...then the corresponding column_number is appended to the time_columns set
        if time_columns: #if there is something in time_columns...
            print("row", row_number, "->", time_columns) #...then print the given text and row_number and columns that time is in


find_header_rows("06JUL26", "Total") #passing the function our two arguments here
find_header_rows("06JUL26", "Time")

find_header_rows("20APR26", "Total")
find_header_rows("20APR26", "Time")

find_header_rows("02Feb26", "Total")
find_header_rows("02Feb26", "Time")


row 26 -> [2, 11]
row 54 -> [2, 11]
row 81 -> [2, 11]
row 108 -> [2]
row 3 -> [2, 11]
row 31 -> [2, 11]
row 58 -> [2, 11]
row 85 -> [2]
row 26 -> [2, 11]
row 54 -> [2, 11]
row 81 -> [2, 11]
row 108 -> [2]
row 3 -> [2, 11]
row 31 -> [2, 11]
row 58 -> [2, 11]
row 85 -> [2]
row 26 -> [2, 11]
row 54 -> [2, 11]
row 81 -> [2, 11]
row 110 -> [2]
row 3 -> [2, 11]
row 31 -> [2, 11]
row 58 -> [2, 11]
row 87 -> [2]


This confirms the locations for both "Time" and "Total", which tells me the layout of the sheet. In row 26, "Total" is in column 2 and 11, and so on. In row 3, "Total" is in column 2 and 11, and so on.

02Feb26 band 4 starts at row 87, not 85. Total at 110, not 108.

Confirms that Era 3 Bands (each multiple of rows containing a given set of days, like Monday+Tuesday, Wednesday+Thursday, etc) are standard on 1-3, but differ for Band 4 (Sunday)

Necessitates the parser to account for this difference, cannot hardcode for 4 header rows as 3, 31, 58, and 85.

Running the function on some era 1 and era 2 sheets.

In [15]:
find_header_rows("2Dec24", "Time")
find_header_rows("2Dec24", "Total")

find_header_rows("13OCT25", "Time")
find_header_rows("13OCT25", "Total")

row 2 -> [1, 6]
row 21 -> [1, 6]
row 40 -> [1, 6]
row 59 -> [1]
row 18 -> [1, 6]
row 37 -> [1, 6]
row 56 -> [1, 6]
row 75 -> [1]
row 3 -> [2, 10]
row 31 -> [2, 10]
row 58 -> [2, 10]
row 87 -> [2]
row 12 -> [21]
row 26 -> [2, 10]
row 54 -> [2, 10]
row 81 -> [2, 10]
row 110 -> [2]


This proves that the parser must be dynamic for multiple of our values. Header rows, columns, block height and band spacing cannot be assumed.

Era 2 carries a stray "Total" at row 12 column 21, outside the block columns. Pair headers to totals by column, not by proximity, and assert four of each per sheet.

In [24]:

def print_header_values(sheet_name,target_row):
    wb = load_workbook("../data/raw/Nutrition141025NEW.xlsx",
                    read_only=True, data_only=True) #we have to reload the workbook here because we can only read each sheet once

    ws = wb[sheet_name] 

    for row_number, cell_values in enumerate(ws.iter_rows(values_only=True),start=1): 
        if row_number == target_row:
            for column_number, value in enumerate(cell_values): 
                print("Row:", row_number,"| Column:", column_number,"| Values:", value)
            break


print_header_values("06JUL26", 3)



    

Row: 3 | Column: 0 | Values: None
Row: 3 | Column: 1 | Values: None
Row: 3 | Column: 2 | Values: Time
Row: 3 | Column: 3 | Values: Food
Row: 3 | Column: 4 | Values: Quantity (g or count)
Row: 3 | Column: 5 | Values: Calories
Row: 3 | Column: 6 | Values: Protein
Row: 3 | Column: 7 | Values: Fiber
Row: 3 | Column: 8 | Values: Carbs
Row: 3 | Column: 9 | Values: Fat
Row: 3 | Column: 10 | Values: None
Row: 3 | Column: 11 | Values: Time
Row: 3 | Column: 12 | Values: Food
Row: 3 | Column: 13 | Values: Quantity (g or count)
Row: 3 | Column: 14 | Values: Calories
Row: 3 | Column: 15 | Values: Protein
Row: 3 | Column: 16 | Values: Fiber
Row: 3 | Column: 17 | Values: Carbs
Row: 3 | Column: 18 | Values: Fat
Row: 3 | Column: 19 | Values: None
Row: 3 | Column: 20 | Values: None
Row: 3 | Column: 21 | Values: None
Row: 3 | Column: 22 | Values: None
Row: 3 | Column: 23 | Values: None
Row: 3 | Column: 24 | Values: None
Row: 3 | Column: 25 | Values: None
Row: 3 | Column: 26 | Values: None
Row: 3 | Column

In [25]:
print_header_values("13OCT25", 3)

Row: 3 | Column: 0 | Values: None
Row: 3 | Column: 1 | Values: None
Row: 3 | Column: 2 | Values: Time
Row: 3 | Column: 3 | Values: Food
Row: 3 | Column: 4 | Values: Calories
Row: 3 | Column: 5 | Values: Protein
Row: 3 | Column: 6 | Values: Fiber
Row: 3 | Column: 7 | Values: Carbs
Row: 3 | Column: 8 | Values: Fat
Row: 3 | Column: 9 | Values: None
Row: 3 | Column: 10 | Values: Time
Row: 3 | Column: 11 | Values: Food
Row: 3 | Column: 12 | Values: Calories
Row: 3 | Column: 13 | Values: Protein
Row: 3 | Column: 14 | Values: Fiber
Row: 3 | Column: 15 | Values: Carbs
Row: 3 | Column: 16 | Values: Fat
Row: 3 | Column: 17 | Values: None
Row: 3 | Column: 18 | Values: None
Row: 3 | Column: 19 | Values: None
Row: 3 | Column: 20 | Values: None
Row: 3 | Column: 21 | Values: None
Row: 3 | Column: 22 | Values: None
Row: 3 | Column: 23 | Values: None
Row: 3 | Column: 24 | Values: None


Era 2

In [26]:
print_header_values("2Dec24", 2)

Row: 2 | Column: 0 | Values: None
Row: 2 | Column: 1 | Values: Time
Row: 2 | Column: 2 | Values: Food
Row: 2 | Column: 3 | Values: Calories
Row: 2 | Column: 4 | Values: Protein
Row: 2 | Column: 5 | Values: None
Row: 2 | Column: 6 | Values: Time
Row: 2 | Column: 7 | Values: Food
Row: 2 | Column: 8 | Values: Calories
Row: 2 | Column: 9 | Values: Protein


Era 1

Differences between the three sheets are now clear, with the labels differing. Still assuming there are 3 Eras, however it is possible there are more than that. Will have to walk each sheet and test for this to properly make a determination.

In [ ]:
def altered_find_header_rows(sheet_name, target="Time"):
    wb = load_workbook("../data/raw/Nutrition141025NEW.xlsx",
                   read_only=True, data_only=True) 

    ws = wb[sheet_name] 

    for row_number, cell_values in enumerate(ws.iter_rows(values_only=True, max_row=15),start=1): #loop walks each row and enumerates it
        stripped_cell_values=[]

        for row_value in cell_values:
            stripped_cell_values.append(str(row_value).strip())
            
        if target in stripped_cell_values: 
            return row_number, cell_values

#This function should return the number of and values and on the row which contains the target. If the row contains no target, it does not return it and continues, if the loop ends without a match, the function returns None.


(3, (None, None, 'Time', 'Food', 'Quantity (g or count)', 'Calories', 'Protein', 'Fiber', 'Carbs', 'Fat', None, 'Time', 'Food', 'Quantity (g or count)', 'Calories', 'Protein', 'Fiber', 'Carbs', 'Fat', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None))
(2, (None, 'Time', 'Food', 'Calories', 'Protein', None, 'Time', 'Food', 'Calories', 'Protein'))
None


In [ ]:
print(altered_find_header_rows("06JUL26"))
print(altered_find_header_rows("2Dec24"))
print(altered_find_header_rows("Weight J"))

In [ ]:
def altered_print_header_values(sheet_name,target_row):
    wb = load_workbook("../data/raw/Nutrition141025NEW.xlsx",
                    read_only=True, data_only=True) #we have to reload the workbook here because we can only read each sheet once

    ws = wb[sheet_name] 

    for row_number, cell_values in enumerate(ws.iter_rows(values_only=True),start=1): 
        if row_number == target_row:
            for column_number, value in enumerate(cell_values): 
                print("Row:", row_number,"| Column:", column_number,"| Values:", value)
            break



In [38]:
not_a_week=[]
is_a_week=[]
#created our two lists, for weeks and non weeks.

for name in wb.sheetnames: #for loop runs, with var name being a given workbook's sheetname
    result=altered_find_header_rows(name) #var result is equal to the result the function running on each sheetname
    if result is None: #if the result is None
        not_a_week.append(name) #the name of the sheet is added to not_a_week
    else:
        is_a_week.append(name)#otherwise it is added to is_a_week

print(len(is_a_week), "logs, and ", len(not_a_week), "not logs")  #prints the text alongside specified and uses len to count the amount of entries in each list.          

print(not_a_week)
print(is_a_week)


90 logs, and  4 not logs
['Weight J', 'Weight J (2)', 'Weight OR', 'Costs']
['Blank Simplified', 'Blank Week', '14Oct24', '4Nov24', 'Sheet1 (2)', '19Nov24', '25Nov24', '2Dec24', '9Dec24', '16Dec24', '30Dec24', '6Jan24', '20Jan24', '27Jan25', '03Feb25', '10Feb25', '17Feb25', '24Feb25', '03Mar25', '10Mar25', '17Mar25', '24Mar25', '31Mar25', '07Apr25', '14Apr25', '21Apr25', '28Apr25', '05May25', '13May25', '19May25', '26May25', '02June25', '09June25', '30JUN25', '07JUL25', '14JUL25', 'Blank Schedule (2)', 'Blank (2)', '11AUG25', '18AUG25', '25AUG25', '01SEP25', '08SEP25', '15SEP25', '06OCT25', '13OCT25', '20OCT25', '27OCT25', '03NOV25', 'IGNORE', '10NOV25', '17NOV25', '24NOV25', '01DEC25', '08DEC25', '29DEC25', '05JAN26', '12JAN26', '19JAN26', '26Jan26', '02Feb26', '09Feb26', '16Feb26', '23Feb26', '02Mar26', '09Mar26', '16Mar26', '23Mar26', '30Mar26', '07APR26', '13APR26', '20APR26', '27APR26', '04MAY26', '11MAY26', '18MAY26', '25MAY26', 'College Sample', 'Blank Sheet Official', '01JUN26'

This tells us the amount of 
This number contains all valid logs, as well as blank templates, unused weekly sheets, and projections.
Cannot rely on this as a final test of valid sheets, only as a test of which sheets are and are not weekly logs of some form.
Blanks and templates are of lowest concern, can be detected. However, projections cannot, and contain synthetic data which could corrupt future results.

Non-real weeks: Blank Simplified, Blank Week, Sheet1 (2), Blank Schedule (2), Blank (2), IGNORE, College Sample, Blank Sheet Official, IGNORE(2)

In [ ]:
r"(\d{1,2})([A-Za-z]+)(\d{2})$" #regular expression we will use to describe the shape of our valid sheet names
#\d is any single digit, 0-9
#{1,2} is one or two of the thing before it, in our case any single digit 0-9
#[A-Za-z] is any single letter, uppercase or lowercase
#+ is one or more of the thing before it (our single letter)
#{2}, exactly 2 of the thing before it (single digit 0-9)
#$ end of the string
#parentheses means to capture this part, allowing us to use it later.
#r makes this into a raw string, which is necessary as regex uses backslashes a lot, which would cause us to run into issues.

#In relation to both Excel and SQL, similar in concept, different in syntax. My familiarity at this stage is low.

In [158]:
import re #importing regular expressions

pattern=r"(\d{1,2})([A-Za-z]+)(\d{2})$" 

refined_weeks=[]

for name in is_a_week:
    nameMatching = re.match(pattern, name.strip())
    if nameMatching is None:
        print(repr(name))
    else: 
        refined_weeks.append(name)

print(len(refined_weeks))





#loop over is_a_week
#similar to is None used earlier, test should be for if the name matches the pattern
#If it does not match, it should be put into a different list
#the pattern needs to be given to the loop so it goes over the whole thing

'Blank Simplified'
'Blank Week'
'Sheet1 (2)'
'Blank Schedule (2)'
'Blank (2)'
'IGNORE'
'College Sample'
'Blank Sheet Official'
'IGNORE(2)'
81


Importing regular expressions, then using it with the given pattern, then using a similar loop from earlier, we walk through each sheet and see if it matches the pattern. If it doesn't, the stripped representation is printed so whitespace doesn't cause a failure. 

If it gets to else, it is put into refined_weeks, our confirmed list of genuine weeks.

In [136]:
from datetime import datetime, timedelta

# %b will allow us to get any abbreviated Month names 
# %B will get any full ones
# %y should parse the 2 digit years.
# %d is day of the month, zero padded like 01, 02
# %-d is day of the month with no zero padding like 1, 2

d_raw="02June25"
pattern=r"(\d{1,2})([A-Za-z]+)(\d{2})$" 
m = re.match(pattern, d_raw)

In [137]:
date_raw="06JUL26"
pattern=r"(\d{1,2})([A-Za-z]+)(\d{2})$" 
m = re.match(pattern, date_raw)

day_test=m.group(1)
month_test=m.group(2)
year_test=m.group(3)

month_normal=month_test[:3]

normalised_date=day_test+month_normal+year_test



In [147]:
def date_normalisation(date_raw):    
    pattern=r"(\d{1,2})([A-Za-z]+)(\d{2})$" #using the pattern from earlier
    m = re.match(pattern, date_raw) #m is the match of the pattern with the given date

    day_test=m.group(1) #gives our day date value in accordance with the pattern
    month_test=m.group(2) #month value
    year_test=m.group(3) #year value

    month_normal=month_test[:3] #slices the month to the first 3 characters

    normalised_date=day_test+month_normal+year_test 

    stripped_normalised_date = datetime.strptime(normalised_date, "%d%b%y") #date_processed is the stripped time of the normalised date with the given pattern
    stripped_normalised_date= stripped_normalised_date-timedelta(days=stripped_normalised_date.weekday())

    return stripped_normalised_date



In [152]:
processed_date=date_normalisation("02June25")
print(processed_date)

week_start=processed_date

2025-06-02 00:00:00


In [ ]:
snap_test_day1=date_normalisation("19May25")
print(snap_test_day1.weekday())

snap_test_day2=date_normalisation("13May25")
print(snap_test_day2.weekday())



0
1


Using the date_normalisation function and weekday(), the printed output is the zero-indexed position of the day of the week for the given date. Added the snap to the function above.

In [ ]:
date_normalisation("13May25")


datetime.datetime(2025, 5, 12, 0, 0)

In [ ]:
refined_dates=[] #creates a list 
for a_sheet in refined_weeks: #starts the loop
    rd=date_normalisation(a_sheet) #rd is the result of the function on the var a_sheet
    refined_dates.append((a_sheet, rd)) #adds the date and its refined to the list
    assert rd.weekday() == 0, f"{a_sheet} snapped to {rd}" #this means if the weekday of any of them are not 0, the function will stop
print(len(refined_dates))
print(refined_dates)

81
[('14Oct24', datetime.datetime(2024, 10, 14, 0, 0)), ('4Nov24', datetime.datetime(2024, 11, 4, 0, 0)), ('19Nov24', datetime.datetime(2024, 11, 18, 0, 0)), ('25Nov24', datetime.datetime(2024, 11, 25, 0, 0)), ('2Dec24', datetime.datetime(2024, 12, 2, 0, 0)), ('9Dec24', datetime.datetime(2024, 12, 9, 0, 0)), ('16Dec24', datetime.datetime(2024, 12, 16, 0, 0)), ('30Dec24', datetime.datetime(2024, 12, 30, 0, 0)), ('6Jan24', datetime.datetime(2024, 1, 1, 0, 0)), ('20Jan24', datetime.datetime(2024, 1, 15, 0, 0)), ('27Jan25', datetime.datetime(2025, 1, 27, 0, 0)), ('03Feb25', datetime.datetime(2025, 2, 3, 0, 0)), ('10Feb25', datetime.datetime(2025, 2, 10, 0, 0)), ('17Feb25', datetime.datetime(2025, 2, 17, 0, 0)), ('24Feb25', datetime.datetime(2025, 2, 24, 0, 0)), ('03Mar25', datetime.datetime(2025, 3, 3, 0, 0)), ('10Mar25', datetime.datetime(2025, 3, 10, 0, 0)), ('17Mar25', datetime.datetime(2025, 3, 17, 0, 0)), ('24Mar25', datetime.datetime(2025, 3, 24, 0, 0)), ('31Mar25', datetime.datetime